# Early Exit Sanity Check

验证 `generate_with_dual_cache_early_exit` 方案A的正确性。

**核心测试**：当 `early_exit_threshold=1.0` 时（没有 token 会被判定为稳定），
应该退化为 `generate_with_dual_cache` 的结果（完全一致）。

**方案A原理**：
- 前 `early_exit_layer` 层正常计算所有 token
- 在 `early_exit_layer` 层后，比较当前 hidden state 与上一 step 同层的 cos_sim
- 如果 cos_sim >= threshold，后续层复用上一 step 的 hidden state
- Attention 仍然全部计算（为了 KV cache 更新）

In [1]:
import torch
import time
from transformers import AutoTokenizer
from model.modeling_llada import LLaDAModelLM
from generate import generate_with_dual_cache, generate_with_dual_cache_early_exit

/home/pianng/miniconda3/envs/dllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 加载模型
device = 'cuda'
model = LLaDAModelLM.from_pretrained('GSAI-ML/LLaDA-8B-Instruct', torch_dtype=torch.bfloat16).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained('GSAI-ML/LLaDA-8B-Instruct')

Loading checkpoint shards: 100%|██████████| 6/6 [00:00<00:00,  8.51it/s]


In [3]:
# 准备输入
prompt = "What is the capital of France?"
m = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
input_ids = torch.tensor(tokenizer(text)['input_ids']).to(device).unsqueeze(0)
print(f"Input length: {input_ids.shape[1]}")

Input length: 20


## Test 1: Sanity Check - threshold=1.0 应该退化为 baseline

In [4]:
# 设置相同的随机种子以确保可重复性
torch.manual_seed(42)

# Baseline: generate_with_dual_cache
start = time.time()
out_baseline, nfe_baseline = generate_with_dual_cache(
    model, input_ids, 
    steps=128, gen_length=128, block_length=32, 
    threshold=0.9
)
t_baseline = time.time() - start
ans_baseline = tokenizer.decode(out_baseline[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Baseline: {t_baseline:.2f}s, NFE={nfe_baseline}")
print(f"Output: {ans_baseline[:200]}...")

Baseline: 1.58s, NFE=7
Output: The capital of France is Paris....


In [5]:
# 重置随机种子
torch.manual_seed(42)

# Early Exit with threshold=1.0 (应该和 baseline 完全一致)
start = time.time()
out_ee, nfe_ee, skip_ratio = generate_with_dual_cache_early_exit(
    model, input_ids,
    steps=128, gen_length=128, block_length=32,
    threshold=0.9,
    early_exit_layer=16,
    early_exit_threshold=1.0,  # 没有 token 会被 skip
)
t_ee = time.time() - start
ans_ee = tokenizer.decode(out_ee[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Early Exit (threshold=1.0): {t_ee:.2f}s, NFE={nfe_ee}, skip_ratio={skip_ratio:.4f}")
print(f"Output: {ans_ee[:200]}...")

Early Exit (threshold=1.0): 0.54s, NFE=7, skip_ratio=0.0000
Output: The capital of France is Paris....


In [ ]:
# 验证输出是否完全一致
print("\n" + "="*60)
print("Sanity Check Results (Prompt 1):")
print("="*60)

tokens_match = torch.all(out_baseline == out_ee).item()
print(f"Token IDs match: {tokens_match}")
print(f"NFE match: {nfe_baseline == nfe_ee}")
print(f"Skip ratio (should be 0.0): {skip_ratio}")

if tokens_match and skip_ratio == 0.0:
    print("\n✓ SANITY CHECK PASSED: threshold=1.0 correctly degrades to baseline")
else:
    print("\n✗ SANITY CHECK FAILED")
    if not tokens_match:
        # 找出不匹配的位置
        diff_mask = out_baseline != out_ee
        diff_positions = diff_mask.nonzero(as_tuple=True)
        print(f"Mismatched positions: {diff_positions}")
        for i in range(min(5, len(diff_positions[0]))):
            pos = diff_positions[1][i].item()
            print(f"  Position {pos}: baseline={out_baseline[0, pos].item()}, early_exit={out_ee[0, pos].item()}")


Sanity Check Results (Prompt 1):
Token IDs match: True
NFE match: True
Skip ratio (should be 0.0): 0.0

✓ SANITY CHECK PASSED: threshold=1.0 correctly degrades to baseline


In [7]:
# ===== 多 Prompt Sanity Check =====
# 测试更多 prompt 确保 threshold=1.0 总是退化为 baseline

sanity_prompts = [
    "What is the capital of France?",
    "Explain quantum mechanics in simple terms.",
    "Write a haiku about mountains.",
    "What is 2 + 2?",
    "Who invented the telephone?",
]

print("="*80)
print("COMPREHENSIVE SANITY CHECK: threshold=1.0 vs baseline")
print("="*80)

all_passed = True
results = []

for idx, prompt in enumerate(sanity_prompts):
    m = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    test_input_ids = torch.tensor(tokenizer(text)['input_ids']).to(device).unsqueeze(0)
    
    # Baseline
    torch.manual_seed(42)
    out_base, nfe_base = generate_with_dual_cache(
        model, test_input_ids, steps=128, gen_length=128, block_length=32, threshold=0.9
    )
    
    # Early Exit with threshold=1.0
    torch.manual_seed(42)
    out_ee, nfe_ee, skip_ratio = generate_with_dual_cache_early_exit(
        model, test_input_ids, steps=128, gen_length=128, block_length=32, threshold=0.9,
        early_exit_layer=16, early_exit_threshold=1.0
    )
    
    # Check
    tokens_match = torch.all(out_base == out_ee).item()
    nfe_match = (nfe_base == nfe_ee)
    skip_zero = (skip_ratio == 0.0)
    passed = tokens_match and nfe_match and skip_zero
    
    status = "✓ PASS" if passed else "✗ FAIL"
    results.append({
        'prompt': prompt[:40] + "..." if len(prompt) > 40 else prompt,
        'tokens_match': tokens_match,
        'nfe_match': nfe_match,
        'skip_zero': skip_zero,
        'passed': passed
    })
    
    if not passed:
        all_passed = False
    
    print(f"\n[{idx+1}/{len(sanity_prompts)}] {status}")
    print(f"  Prompt: {prompt[:50]}...")
    print(f"  Tokens match: {tokens_match}, NFE match: {nfe_match}, Skip=0: {skip_zero}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"{'Prompt':<45} {'Tokens':<10} {'NFE':<8} {'Skip=0':<10} {'Status'}")
print("-"*80)
for r in results:
    status = "✓" if r['passed'] else "✗"
    print(f"{r['prompt']:<45} {str(r['tokens_match']):<10} {str(r['nfe_match']):<8} {str(r['skip_zero']):<10} {status}")

print("\n" + "="*80)
if all_passed:
    print("✓✓✓ ALL SANITY CHECKS PASSED ✓✓✓")
    print("threshold=1.0 correctly degrades to baseline for all test cases!")
else:
    print("✗✗✗ SOME SANITY CHECKS FAILED ✗✗✗")
    print("Please investigate the failed cases above.")
print("="*80)

COMPREHENSIVE SANITY CHECK: threshold=1.0 vs baseline

[1/5] ✓ PASS
  Prompt: What is the capital of France?...
  Tokens match: True, NFE match: True, Skip=0: True

[2/5] ✓ PASS
  Prompt: Explain quantum mechanics in simple terms....
  Tokens match: True, NFE match: True, Skip=0: True

[3/5] ✓ PASS
  Prompt: Write a haiku about mountains....
  Tokens match: True, NFE match: True, Skip=0: True

[4/5] ✓ PASS
  Prompt: What is 2 + 2?...
  Tokens match: True, NFE match: True, Skip=0: True

[5/5] ✓ PASS
  Prompt: Who invented the telephone?...
  Tokens match: True, NFE match: True, Skip=0: True

SUMMARY
Prompt                                        Tokens     NFE      Skip=0     Status
--------------------------------------------------------------------------------
What is the capital of France?                True       True     True       ✓
Explain quantum mechanics in simple term...   True       True     True       ✓
Write a haiku about mountains.                True       True     True 

## Test 2: 新增保护机制测试

测试两个新功能：
1. **上轮 skip 的 token 这轮必须重算**（防止误差累积）
2. **每 K 步强制全算**（force_full_every_k=4）

In [8]:
# 测试不同的 force_full_every_k 设置
print("="*80)
print("测试 force_full_every_k（每 K 步强制全算）")
print("="*80)
print(f"{'force_k':<12} {'Threshold':<12} {'Time(s)':<10} {'NFE':<8} {'Skip%':<10} {'Output Preview'}")
print("-"*80)

# 固定 threshold=0.95，测试不同的 force_k
for force_k in [0, 2, 4, 8]:
    torch.manual_seed(42)
    start = time.time()
    out, nfe, skip_ratio = generate_with_dual_cache_early_exit(
        model, input_ids,
        steps=128, gen_length=128, block_length=32,
        threshold=0.9,
        early_exit_layer=16,
        early_exit_threshold=0.95,
        force_full_every_k=force_k,  # 新增参数
    )
    t = time.time() - start
    ans = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    has_chinese = any('\u4e00' <= c <= '\u9fff' for c in ans)
    quality = "🇨🇳" if has_chinese else "✓"
    print(f"{force_k:<12} {0.95:<12} {t:<10.2f} {nfe:<8} {skip_ratio*100:<10.2f} {quality} {ans[:40]}...")

测试 force_full_every_k（每 K 步强制全算）
force_k      Threshold    Time(s)    NFE      Skip%      Output Preview
--------------------------------------------------------------------------------
0            0.95         0.52       7        11.46      ✓ The capital of France is Paris....
2            0.95         0.49       7        7.81       ✓ The capital of France is Paris....
4            0.95         0.50       7        11.46      ✓ The capital of France is Paris....
8            0.95         0.50       7        11.46      ✓ The capital of France is Paris....


In [9]:
# 测试不同 threshold + force_k=4 的效果
print("\n" + "="*80)
print("不同 threshold + force_full_every_k=4")
print("="*80)
print(f"{'Threshold':<12} {'Time(s)':<10} {'NFE':<8} {'Skip%':<10} {'Quality':<8} {'Output Preview'}")
print("-"*80)

for thresh in [1.0, 0.99, 0.98, 0.97, 0.96, 0.95]:
    torch.manual_seed(42)
    start = time.time()
    out, nfe, skip_ratio = generate_with_dual_cache_early_exit(
        model, input_ids,
        steps=128, gen_length=128, block_length=32,
        threshold=0.9,
        early_exit_layer=16,
        early_exit_threshold=thresh,
        force_full_every_k=4,
    )
    t = time.time() - start
    ans = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    has_chinese = any('\u4e00' <= c <= '\u9fff' for c in ans)
    quality = "🇨🇳" if has_chinese else "✓"
    print(f"{thresh:<12} {t:<10.2f} {nfe:<8} {skip_ratio*100:<10.2f} {quality:<8} {ans[:40]}...")


不同 threshold + force_full_every_k=4
Threshold    Time(s)    NFE      Skip%      Quality  Output Preview
--------------------------------------------------------------------------------
1.0          0.55       7        0.00       ✓        The capital of France is Paris....
0.99         0.52       7        0.00       ✓        The capital of France is Paris....
0.98         0.56       7        0.00       ✓        The capital of France is Paris....
0.97         0.54       7        2.08       ✓        The capital of France is Paris....
0.96         0.47       7        4.17       ✓        The capital of France is Paris....
0.95         0.48       7        11.46      ✓        The capital of France is Paris....


## Test 2.5: 保护措施验证

验证两个保护措施的代码路径是否正确工作：
1. **上轮 skip 的这轮必须重算** - 需要在 threshold < 1.0 时测试
2. **每 K 步强制全算** - 比较 force_k=1 和 baseline

In [10]:
# 验证保护措施 1: force_full_every_k=1 应该等于 baseline
# 因为每步都强制全算，等于完全不 skip

print("="*80)
print("保护措施验证: force_full_every_k=1 应该等于 baseline")
print("="*80)

prompt = "What is the capital of France?"
m = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
test_ids = torch.tensor(tokenizer(text)['input_ids']).to(device).unsqueeze(0)

# Baseline
torch.manual_seed(42)
out_base, nfe_base = generate_with_dual_cache(
    model, test_ids, steps=64, gen_length=64, block_length=32, threshold=0.9
)

# Early Exit with threshold=0.9 but force_k=1 (每步都强制全算)
torch.manual_seed(42)
out_ee, nfe_ee, skip_ratio = generate_with_dual_cache_early_exit(
    model, test_ids, steps=64, gen_length=64, block_length=32, threshold=0.9,
    early_exit_layer=16,
    early_exit_threshold=0.9,  # 低 threshold，正常情况会 skip 很多
    force_full_every_k=1,      # 但每步都强制全算
)

tokens_match = torch.all(out_base == out_ee).item()
print(f"Baseline NFE: {nfe_base}")
print(f"Early Exit (thresh=0.9, force_k=1) NFE: {nfe_ee}")
print(f"Tokens match: {tokens_match}")
print(f"Skip ratio: {skip_ratio} (should be 0.0 because force_k=1)")

if tokens_match and skip_ratio == 0.0:
    print("\n✓ 保护措施验证通过: force_full_every_k=1 正确退化为 baseline")
else:
    print("\n✗ 保护措施验证失败!")
    ans_base = tokenizer.decode(out_base[0, test_ids.shape[1]:], skip_special_tokens=True)
    ans_ee = tokenizer.decode(out_ee[0, test_ids.shape[1]:], skip_special_tokens=True)
    print(f"Baseline: {ans_base[:100]}")
    print(f"EarlyExit: {ans_ee[:100]}")

保护措施验证: force_full_every_k=1 应该等于 baseline
Baseline NFE: 5
Early Exit (thresh=0.9, force_k=1) NFE: 5
Tokens match: True
Skip ratio: 0.0 (should be 0.0 because force_k=1)

✓ 保护措施验证通过: force_full_every_k=1 正确退化为 baseline


In [11]:
# 验证保护措施的效果：比较不同保护级别的输出质量
# force_k=0 (无强制全算) vs force_k=4 (每4步强制) vs force_k=2 (每2步强制)

print("\n" + "="*80)
print("保护措施效果对比: 不同 force_k 对输出质量的影响")
print("="*80)

prompt = "Explain the theory of relativity in simple terms."
m = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
test_ids = torch.tensor(tokenizer(text)['input_ids']).to(device).unsqueeze(0)

# Baseline
torch.manual_seed(42)
out_base, nfe_base = generate_with_dual_cache(
    model, test_ids, steps=64, gen_length=64, block_length=32, threshold=0.9
)
ans_base = tokenizer.decode(out_base[0, test_ids.shape[1]:], skip_special_tokens=True)
print(f"\n[Baseline] NFE={nfe_base}")
print(f"  Output: {ans_base[:150]}...")

# 测试不同 force_k
for force_k in [0, 4, 2, 1]:
    torch.manual_seed(42)
    out, nfe, skip = generate_with_dual_cache_early_exit(
        model, test_ids, steps=64, gen_length=64, block_length=32, threshold=0.9,
        early_exit_layer=16, early_exit_threshold=0.95, force_full_every_k=force_k
    )
    ans = tokenizer.decode(out[0, test_ids.shape[1]:], skip_special_tokens=True)
    
    # 计算与 baseline 的 token 匹配率
    match_rate = (out_base == out).float().mean().item() * 100
    has_chinese = any('\u4e00' <= c <= '\u9fff' for c in ans)
    quality = "🇨🇳 乱码" if has_chinese else "✓ 正常"
    
    print(f"\n[force_k={force_k}] NFE={nfe}, skip={skip*100:.1f}%, match={match_rate:.1f}% {quality}")
    print(f"  Output: {ans[:150]}...")


保护措施效果对比: 不同 force_k 对输出质量的影响

[Baseline] NFE=60
  Output: The theory of relativity states that space and time are relative and can change depending on an observer's state of motion. It means that time passes ...

[force_k=0] NFE=57, skip=26.3%, match=62.8% ✓ 正常
  Output: The theory of relativity states that space and time are relative and can change depending on the observer's state of motion. It means that time passes...

[force_k=4] NFE=60, skip=30.3%, match=96.5% ✓ 正常
  Output: The theory of relativity states that space and time are relative and can change depending on an observer's state of motion. It means that time passes ...

[force_k=2] NFE=62, skip=44.7%, match=84.9% ✓ 正常
  Output: The theory of relativity states that space and time are relative and can change depending on an observer's state of motion. It means that time passes ...

[force_k=1] NFE=60, skip=0.0%, match=100.0% ✓ 正常
  Output: The theory of relativity states that space and time are relative and can change depe

## Test 3: 不同 early_exit_layer 的效果

In [12]:
# 测试不同的 early_exit_layer（判定点）
layers = [8, 12, 16, 20, 24]
threshold = 0.95

print("\n" + "="*80)
print(f"Early Exit Layer comparison (threshold={threshold})")
print(f"{'Layer':<12} {'Time(s)':<10} {'NFE':<8} {'Skip%':<10} {'Output Preview'}")
print("="*80)

for layer in layers:
    torch.manual_seed(42)
    start = time.time()
    out, nfe, skip_ratio = generate_with_dual_cache_early_exit(
        model, input_ids,
        steps=128, gen_length=128, block_length=32,
        threshold=0.9,
        early_exit_layer=layer,
        early_exit_threshold=threshold,
    )
    t = time.time() - start
    ans = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    print(f"{layer:<12} {t:<10.2f} {nfe:<8} {skip_ratio*100:<10.2f} {ans[:50]}...")


Early Exit Layer comparison (threshold=0.95)
Layer        Time(s)    NFE      Skip%      Output Preview
8            2.15       27       61.11      The capital of France is Paris. It is one of the m...
12           0.59       8        45.83      The capital of France is Paris....
16           0.55       7        11.46      The capital of France is Paris....
20           0.54       7        8.33       The capital of France is Paris....
24           0.48       7        3.12       The capital of France is Paris....


## Test 4: 与 Baseline 的质量对比

In [13]:
# 用更长的 prompt 测试
prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a short poem about the ocean.",
    "What are the main causes of climate change?",
]

for prompt in prompts:
    print("\n" + "="*80)
    print(f"Prompt: {prompt}")
    print("="*80)
    
    m = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(text)['input_ids']).to(device).unsqueeze(0)
    
    # Baseline
    torch.manual_seed(42)
    out_base, nfe_base = generate_with_dual_cache(
        model, input_ids, steps=64, gen_length=64, block_length=32, threshold=0.9
    )
    ans_base = tokenizer.decode(out_base[0, input_ids.shape[1]:], skip_special_tokens=True)
    
    # Early Exit (threshold=0.95)
    torch.manual_seed(42)
    out_ee, nfe_ee, skip_ratio = generate_with_dual_cache_early_exit(
        model, input_ids, steps=64, gen_length=64, block_length=32, threshold=0.9,
        early_exit_layer=16, early_exit_threshold=0.95
    )
    ans_ee = tokenizer.decode(out_ee[0, input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"\nBaseline (NFE={nfe_base}):")
    print(ans_base)
    print(f"\nEarly Exit (NFE={nfe_ee}, skip={skip_ratio*100:.1f}%):")
    print(ans_ee)
    
    # 计算 token 重合率
    match_ratio = (out_base == out_ee).float().mean().item()
    print(f"\nToken match ratio: {match_ratio*100:.2f}%")


Prompt: Explain the theory of relativity in simple terms.

Baseline (NFE=60):
The theory of relativity states that space and time are relative and can change depending on an observer's state of motion. It means that time passes differently for observers in motion, and objects can appear longer or shorter depending on their speed. This theory revolutionized our understanding of the universe and our own place within it.

Early Exit (NFE=60, skip=30.3%):
The theory of relativity states that space and time are relative and can change depending on an observer's state of motion. It means that time passes differently for observers in motion, and objects can appear longer or shorter depending on their speed. This theory revolutionized our understanding of the universe and humanity's place in it.

Token match ratio: 96.51%

Prompt: Write a short poem about the ocean.

Baseline (NFE=63):
The ocean's endless blue,
A world of mystery and might,
Its waves that crash and roar,
A symphony of sound a

In [14]:
print("\nDone!")


Done!
